<a href="https://colab.research.google.com/github/mugalan/classical-mechanics-from-a-geometric-point-of-view/blob/main/mechanics/answers-to-selected-assignments/Answers_to_Assignment_on_Free_Rigid_Body_Motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
import numpy as np
import scipy as sp
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy

from sympy import symbols
from sympy import *
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
init_vprinting(pretty_print=True)

import plotly.graph_objects as go

In [ ]:
# Install the simulation helpers
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

# Assignment on Free Rigid Body Motion

Consider a uniform solid rectangular block of mass
$M=10\,\mathrm{kg}$, length $l=2\,\mathrm{m}$,
width $w=1\,\mathrm{m}$, and height $h=4\,\mathrm{m}$,
moving with no external forces or force moments.

Let $\mathbf e$ be an inertial frame, and let
$\mathbf b=\mathbf eR$ be a body-fixed frame with
origin at the center of mass and axes aligned with
the length, width, and height of the block, respectively.
These are principal axes, and the inertia tensor is
$$
\mathbb{I}
=\frac{M}{12}\operatorname{diag}
\left(w^2+h^2,\ l^2+h^2,\ l^2+w^2\right)
=\operatorname{diag}
\left(\frac{85}{6},\frac{50}{3},\frac{25}{6}\right)
\,\mathrm{kg\,m^2}.
$$
Let $\Pi=\mathbb{I}\Omega$ be the body representation
of angular momentum about the center of mass.
Answer the following:

* Write down the rotational equations of motion
using $(R,\Pi)$ as the variables. Show that
$$
\pi=R\Pi,\qquad
E=\frac12\Pi^T\mathbb{I}^{-1}\Pi,\qquad
h^2=\Pi^T\Pi
$$
are conserved.

* Identify the steady rotations about the three
principal axes.

* Simulate the rotational motion
with $R(0)=I_{3\times3}$
$$
\Omega(0)=\Omega_0
\begin{bmatrix}1\\\varepsilon\\\varepsilon\end{bmatrix},
\qquad
\Omega(0)=\Omega_0
\begin{bmatrix}\varepsilon\\1\\\varepsilon\end{bmatrix},
\qquad
\Omega(0)=\Omega_0
\begin{bmatrix}\varepsilon\\\varepsilon\\1\end{bmatrix},
$$
where $\Omega_0=5\,\mathrm{s^{-1}}$ and
$\varepsilon=0.05$. Use $\Pi(0)=\mathbb{I}\Omega(0)$
and simulate for at least $60\,\mathrm{s}$.

* For each initial condition:

    - Plot the intersection of the body momentum sphere $\|\Pi(t)\|=h$ and the the Kinetic energy ellipsoid $E=\frac12\Pi^T\mathbb{I}^{-1}\Pi$

    - Animate the block with arrows representing $\pi(t)$, $\Pi(t)$, $\omega(t)$, and $\Omega(t)$ and their tip paths. Use the display switches to compare selected quantities, distinguishing the spatial representations $\pi$ and $\omega$ from the body-component overlays $\Pi$ and $\Omega$.

    - Explain why $\pi$ remains constant while $\Pi$ generally varies, why angular velocity need not align with angular momentum, and why constant body angular momentum does not imply a stationary body orientation.

* Plot the nutation angle vs time.

* Linearize the body angular momentum equations about
each nonzero steady rotation. Determine which rotations
have oscillatory transverse perturbations and which
have exponentially growing transverse perturbations.

* Compare your simlations with the behavior of the [t-handle in outer space:](https://youtu.be/1n-HMSCDYtM?si=DFHNzvimpn0UXx3w)

#Sample Answers

## Equations of motion in the spatial frame

\begin{align*}
\dot{R}
&=\widehat{\omega}R,
\qquad \omega=\mathbb{I}_R^{-1}\pi\quad \mathbb{I}_R=R\mathbb I R^T,\\
\dot{\pi}
&=0
\end{align*}


From this it follows that:

* The spatial angular momentum $\pi(t)=\mu$ is constant.
* Since
$$\dfrac{d}{dt}KE=T_e^T\Omega=0$$
we see that the total kinetic energy is constant; $KE=E=\mathrm{constant}$.
* Since $\|\Pi(t)\|=\|\pi(t)\|=\|\mu\|=h=\mathrm{constant}$

## Equations of motion in the body frame

\begin{align*}
\dot{R}
&=R\,\widehat{\mathbb{I}^{-1}\Pi},\\
\dot{\Pi}
&=\Pi\times(\mathbb{I}^{-1}\Pi).
\end{align*}


## The Relative Equilibria

When $\mathbb{I}$ is diagonal, these equations admit the equilibrium solutions
$$
\Pi(t)\equiv\pm h
\begin{bmatrix}1\\0\\0\end{bmatrix},
\qquad
\Pi(t)\equiv\pm h
\begin{bmatrix}0\\1\\0\end{bmatrix},
\qquad
\Pi(t)\equiv\pm h
\begin{bmatrix}0\\0\\1\end{bmatrix}.
$$
Since $\Omega=\mathbb{I}^{-1}\Pi$, these solutions
correspond to steady rotations about the principal
axes. They are equilibria of the body angular momentum
equations, but the orientation $R$ continues to change.
They are therefore called *relative equilibria*
of the rigid body motion. When the principal moments
are distinct, these are the only equilibria on
$\|\Pi\|=h$.

## Setting the Parameters and Initial Conditions

In [ ]:
#### Defining the parameters
M=10;l=2;w=1;h=4; II_1=M*(w**2+h**2)/12; II_2=M*(l**2+h**2)/12;II_3=M*(w**2+l**2)/12;
II=np.array([[II_1,0.,0.],[0.,II_2,0.],[0.,0.,II_3]])
cubeDimensions={'l':l,'w':w,'h':h,'xp':l/2,'yp':w/2,'zp':h/2,};
parameters={'CM':np.array([l/2-cubeDimensions['xp'],w/2-cubeDimensions['yp'],h/2-cubeDimensions['zp']]), 'g':10, 'M':M, 'II':II};

In [ ]:
#### Definig the initial conditions
ICq=mr.q_from_axis_angles(0*np.pi/18,np.array([1,0,0])); ICR=mr.r_from_quaternions(ICq);
ICOmega=np.array([5.0,.5,0.0]); ICo=np.array([0.,0.,0.]); ICdoto=np.array([0.,0.,0.]);
ICXC=np.array([0,0,0]);
ICs=[[ICR,ICo],ICR@ICOmega,ICdoto,ICXC]

## Intersection of the Angular Momentum sphere with the Kinetic Energy ellipsoid

In [ ]:
fig_intersection, info = mr.plot_momentum_energy_intersection(
    parameters=parameters,
    ICOmega=ICOmega,
    ICR=ICR,
    overlay_spatial=False,
    sphere_opacity=0.20,
    ellipsoid_opacity=0.25,
    title="Angular-momentum sphere and energy ellipsoid",
    show=True,
)

## Simulating the dynamics

#### Defining the force model

In [ ]:
def externalForceModel(qq,parameters,X):
  #Heavy Top Object
  M=parameters['M']; II=parameters['II']; g=parameters['g']; CM=parameters['CM'];
  R=X[0][0]; omega=X[1]; spi=R@ II@ R.T @ omega
  fe=np.array([0,0,0]);
  taue=np.array([0,0,0]);
  return [taue,fe]

def actuator(qq,parameters, t, X, taue,fe):
  tauu=np.array([0,0,0]);
  fu=np.array([0,0,0]);
  return [tauu,fu]
mr.set_external_force_model(externalForceModel)
mr.set_actuator(actuator)

#### Simulating the dynamics

In [ ]:
dt=0.01; Tmax=20.;
aa4=mr.simulating_a_cube(dt, Tmax, cubeDimensions, parameters,ICs)

In [ ]:
fig = mr.animated_cube_with_momentum(
    cubeDimensions,
    parameters,
    figTitle="Free rigid body: angular momentum and angular velocity",
    show_trails=True,
    body_opacity=1.0,
    duration=20,
    stride=2,
    trail_stride=10,    # Store every twentieth sample in each trail
)

## Plotting the nutation angle

In [ ]:
Xs=mr.trajectory

In [ ]:
II=parameters['II'];

In [ ]:
alphat = []

for X in Xs:
    R = np.asarray(X[0][0])
    omega = np.asarray(X[1])       # Spatial angular velocity

    Omega = R.T @ omega           # Body angular velocity
    Pi = II @ Omega               # Body angular momentum
    h = np.linalg.norm(Pi)

    alpha = (
        np.arccos(np.clip(Pi[2] / h, -1.0, 1.0))
        if h > 0 else np.nan
    )
    alphat.append(alpha)

alphat = np.asarray(alphat)        # Radians
t = dt * np.arange(len(Xs))

In [ ]:
fig = go.Figure(data=go.Scatter(x=t, y=alphat, mode='markers'))
fig.show()

## Linearized Angular Momentum Equations

Let the body frame consist of principal axes, so that
$\mathbb{I}=\operatorname{diag}
(\mathbb{I}_1,\mathbb{I}_2,\mathbb{I}_3)$.
For torque-free motion, the body angular momentum satisfies
\begin{align*}
\dot{\Pi}_1
&=\left(\frac{1}{\mathbb{I}_3}
       -\frac{1}{\mathbb{I}_2}\right)\Pi_2\Pi_3,\\
\dot{\Pi}_2
&=\left(\frac{1}{\mathbb{I}_1}
       -\frac{1}{\mathbb{I}_3}\right)\Pi_3\Pi_1,\\
\dot{\Pi}_3
&=\left(\frac{1}{\mathbb{I}_2}
       -\frac{1}{\mathbb{I}_1}\right)\Pi_1\Pi_2.
\end{align*}
For distinct principal moments, every nonzero steady
rotation has $\Pi=he_i$, where $h\ne0$ is a signed
constant and $e_i$ is a principal-axis coordinate column.
The corresponding angular velocity is
$\Omega=(h/\mathbb{I}_i)e_i$.

To linearize about a steady rotation, write
$\Pi=he_i+\delta\Pi$ and retain only terms that are
first order in $\delta\Pi$. The two perturbation
components perpendicular to $e_i$ are called the
**transverse perturbations**.


### Rotation about the first principal axis

Writing
$$
\Pi=
\begin{bmatrix}
\pm h+\delta\Pi_1\\
\delta\Pi_2\\
\delta\Pi_3
\end{bmatrix},
$$
gives
\begin{align*}
\delta\dot{\Pi}_1&=0,\\
\delta\dot{\Pi}_2
&= \pm h\left(\frac{1}{\mathbb{I}_1}
        -\frac{1}{\mathbb{I}_3}\right)\delta\Pi_3,\\
\delta\dot{\Pi}_3
&=\pm h\left(\frac{1}{\mathbb{I}_2}
        -\frac{1}{\mathbb{I}_1}\right)\delta\Pi_2.
\end{align*}
Differentiating the second equation and substituting
the third gives
$$
\delta\ddot{\Pi}_2
=
\frac{h^2(\mathbb{I}_3-\mathbb{I}_1)
          (\mathbb{I}_1-\mathbb{I}_2)}
     {\mathbb{I}_1^2\mathbb{I}_2\mathbb{I}_3}
\,\delta\Pi_2.
$$
The same second-order equation holds for
$\delta\Pi_3$.



### Rotation about the second principal axis

Writing
$$
\Pi=
\begin{bmatrix}
\delta\Pi_1\\
h+\delta\Pi_2\\
\delta\Pi_3
\end{bmatrix},
$$
gives
\begin{align*}
\delta\dot{\Pi}_1
&=h\left(\frac{1}{\mathbb{I}_3}
        -\frac{1}{\mathbb{I}_2}\right)\delta\Pi_3,\\
\delta\dot{\Pi}_2&=0,\\
\delta\dot{\Pi}_3
&=h\left(\frac{1}{\mathbb{I}_2}
        -\frac{1}{\mathbb{I}_1}\right)\delta\Pi_1.
\end{align*}
Consequently, each transverse component satisfies
$$
\delta\ddot{\Pi}_j
=
\frac{h^2(\mathbb{I}_2-\mathbb{I}_3)
          (\mathbb{I}_1-\mathbb{I}_2)}
     {\mathbb{I}_2^2\mathbb{I}_1\mathbb{I}_3}
\,\delta\Pi_j,
\qquad j=1,3.
$$



### Rotation about the third principal axis

Writing
$$
\Pi=
\begin{bmatrix}
\delta\Pi_1\\
\delta\Pi_2\\
\pm h+\delta\Pi_3
\end{bmatrix},
$$
gives
\begin{align*}
\delta\dot{\Pi}_1
&=\pm h\left(\frac{1}{\mathbb{I}_3}
        -\frac{1}{\mathbb{I}_2}\right)\delta\Pi_2,\\
\delta\dot{\Pi}_2
&=\pm h\left(\frac{1}{\mathbb{I}_1}
        -\frac{1}{\mathbb{I}_3}\right)\delta\Pi_1,\\
\delta\dot{\Pi}_3&=0.
\end{align*}
Consequently, each transverse component satisfies
$$
\delta\ddot{\Pi}_j
=
\frac{h^2(\mathbb{I}_2-\mathbb{I}_3)
          (\mathbb{I}_3-\mathbb{I}_1)}
     {\mathbb{I}_3^2\mathbb{I}_1\mathbb{I}_2}
\,\delta\Pi_j,
\qquad j=1,2.
$$



### Interpretation

Each transverse equation has the form
$$
\ddot z=c_i z.
$$
If $c_i<0$, setting $\nu_i=\sqrt{-c_i}$ gives
$$
z(t)=A\cos(\nu_i t)+B\sin(\nu_i t),
$$
so the linearized transverse perturbations are
bounded and oscillatory.

If $c_i>0$, setting $\sigma_i=\sqrt{c_i}$ gives
$$
z(t)=Ae^{\sigma_i t}+Be^{-\sigma_i t}.
$$
Thus there is an exponentially growing perturbation
direction, and the steady rotation is unstable.

The coefficients depend on $h^2$, so the classification
is unchanged when the direction of steady rotation
is reversed. The longitudinal perturbation is constant
to first order; it changes the magnitude of the steady
rotation. On the fixed angular momentum sphere
$\|\Pi\|=h$, this longitudinal perturbation vanishes
to first order.

Steady rotations about the axes of largest and smallest
principal moments have oscillatory transverse
perturbations. Steady rotations about the intermediate
principal axis have an exponentially growing
transverse perturbation direction.

The oscillatory linearization alone does not prove
nonlinear stability. For the torque-free rigid body,
nonlinear stability of the largest- and smallest-moment
rotations also follows from conservation of kinetic
energy and angular momentum magnitude: these rotations
are strict extrema of kinetic energy on the angular
momentum sphere.

### Application to the rectangular block

For a uniform rectangular block,
$$
\mathbb{I}_1=\frac{M}{12}(w^2+h^2),\qquad
\mathbb{I}_2=\frac{M}{12}(l^2+h^2),\qquad
\mathbb{I}_3=\frac{M}{12}(l^2+w^2).
$$
For the dimensions in this exercise, $h>l>w$, so
$\mathbb{I}_3<\mathbb{I}_1<\mathbb{I}_2$.

Let $s=H/\mathbb{I}_i$ be the signed angular speed
of the steady rotation about the axis under consideration.
Substituting the expressions for the principal moments
into the linearized equations gives
\begin{align*}
\text{About }\mathbf b_1:\qquad
\delta\ddot{\Pi}_j
&=
\frac{(h^2-l^2)(l^2-w^2)}
     {(l^2+h^2)(l^2+w^2)}
s^2\delta\Pi_j,
&&j=2,3,\\
\text{About }\mathbf b_2:\qquad
\delta\ddot{\Pi}_j
&=-
\frac{(h^2-w^2)(l^2-w^2)}
     {(w^2+h^2)(l^2+w^2)}
s^2\delta\Pi_j,
&&j=1,3,\\
\text{About }\mathbf b_3:\qquad
\delta\ddot{\Pi}_j
&=-
\frac{(h^2-w^2)(h^2-l^2)}
     {(w^2+h^2)(l^2+h^2)}
s^2\delta\Pi_j,
&&j=1,2.
\end{align*}
Notice that the common mass factor $M/12$ cancels.

For $l=2\,\mathrm{m}$, $w=1\,\mathrm{m}$, and
$h=4\,\mathrm{m}$, the three dimensionless coefficients are
\begin{align*}
\frac{(16-4)(4-1)}{(4+16)(4+1)}
&=\frac{36}{100}=\frac{9}{25},\\
-\frac{(16-1)(4-1)}{(1+16)(4+1)}
&=-\frac{45}{85}=-\frac{9}{17},\\
-\frac{(16-1)(16-4)}{(1+16)(4+16)}
&=-\frac{180}{340}=-\frac{9}{17}.
\end{align*}
Thus rotation about $\mathbf b_1$ has an exponentially
growing transverse perturbation with growth rate
$$
\sigma_1
=|s|\sqrt{
\frac{(h^2-l^2)(l^2-w^2)}
     {(l^2+h^2)(l^2+w^2)}
}
=\frac35|s|.
$$
The transverse oscillation frequencies about the
other two axes are
\begin{align*}
\nu_2
&=|s|\sqrt{
\frac{(h^2-w^2)(l^2-w^2)}
     {(w^2+h^2)(l^2+w^2)}
}
=\frac{3}{\sqrt{17}}|s|,\\
\nu_3
&=|s|\sqrt{
\frac{(h^2-w^2)(h^2-l^2)}
     {(w^2+h^2)(l^2+h^2)}
}
=\frac{3}{\sqrt{17}}|s|.
\end{align*}
The equality of these two frequencies at the same
$|s|$ is specific to the chosen dimensions; it does
not hold for every rectangular block.

## Interesting Videos

Compare the simlations with the behavior of the

*   [Flipping a tennis racket:](https://youtu.be/1VPfZ_XzisU?si=r8Mcp8JBO7wUtLU0)
*   [T-handle in outer space:](https://youtu.be/1n-HMSCDYtM?si=DFHNzvimpn0UXx3w)